# P1: Data Generation
**ATRD — Adaptive Test-Time Reasoning Distillation**

Phase 1: Baseline evaluation → Failure extraction → Synthetic generation

- Model: `nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-Base-BF16`
- Deliverable: Filtered synthetic SFT dataset

In [ ]:
# Cell 1: Imports and Reproducibility Setup
import random
import numpy as np
import torch
import os, sys, json, re, hashlib
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Configuration
import os
from dataclasses import dataclass
from pathlib import Path

_DEFAULT_OUT = (
    Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path(os.environ.get("ATRD_OUTPUT_DIR", "data"))
)

@dataclass(frozen=True)
class Phase1Config:
    BASE_MODEL: str = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-Base-BF16"
    MAX_TOKENS: int = 7680
    TEMPERATURE: float = 0.0
    BENCHMARK_PATH: str = "/kaggle/input/nemotron-benchmark"
    OUTPUT_DIR: Path = _DEFAULT_OUT
    RAW_SYNTHETIC_PATH: str = str(_DEFAULT_OUT / "raw_synthetic_dataset.jsonl")
    FILTERED_PATH: str = str(_DEFAULT_OUT / "filtered_synthetic_dataset.jsonl")
    FINAL_PATH: str = str(_DEFAULT_OUT / "final_train_dataset.jsonl")
    SYNTHETIC_TARGET: int = 10000
    NUM_FAILURE_MODES: int = 5
    API_MODEL: str = "deepseek-ai/DeepSeek-R1"
    API_TEMPERATURE: float = 0.7

config = Phase1Config()
print(f"Config initialized. Target volume: {config.SYNTHETIC_TARGET} examples.")

In [ ]:
# Cell 3: Helper Functions
import sys
sys.path.insert(0, ".")
import re
from typing import Dict, List, Any, Optional

def format_prompt(question: str) -> str:
    """Format question for baseline generation."""
    return f"Question: {question}\nProvide a complete step-by-step thinking trace inside <<thinking>>...</thinking>> and the final answer in \\boxed{{}}.\n"

from src.evaluation.metric import extract_boxed_answer, answers_equivalent

def check_answer(predicted: str, expected: str, tolerance: float = 0.01) -> bool:
    """Competition-aligned answer check (fractions + relative tolerance)."""
    return answers_equivalent(predicted, expected, tolerance)

def classify_failure(response: Dict[str, Any]) -> str:
    """Classify failure type in model response."""
    answer = response.get('answer', '')
    reasoning = response.get('reasoning', '')
    if not answer:
        return 'no_answer'
    if not reasoning:
        return 'incomplete'
    if '\\boxed' not in answer:
        return 'format_error'
    return 'wrong_answer'

print('Helper functions loaded.')

In [ ]:
# Cell 4: Load Base Model
import sys
sys.path.append('.') # Add workspace root to sys.path

from src.models.loader import ModelLoader

print("Initializing ModelLoader...")
loader = ModelLoader("configs/competition_params.json")
tokenizer = loader.load_tokenizer()
print("Loading base Nemotron model in 4-bit (QLoRA standard)...")
try:
    model = loader.load_model(quantize=True)
    loader.enable_gradient_checkpointing(model)
    print(f"Model loaded: {model.num_parameters():,} params")
except Exception as e:
    print(f"Skipped actual loading (running outside GPU cluster or local system): {e}")
    model = None

In [ ]:
# Cell 5: Baseline Evaluation
import json
from pathlib import Path
from src.evaluation.metric import evaluate_submission

# Load benchmark — Kaggle input or local data/public_test.jsonl
from src.evaluation.metric import load_benchmark_problems

problems = load_benchmark_problems(kaggle_dir=config.BENCHMARK_PATH)
print(f"Loaded {len(problems)} benchmark problems")

# Run baseline model evaluation
if model is None:
    raise RuntimeError("Model must be loaded for baseline evaluation")

responses = []
for p in problems:
    prompt = format_prompt(p["question"])
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256)
    response_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    responses.append({
        "problem_id": p["id"],
        "response": response_text,
        "answer": extract_boxed_answer(response_text),
        "reasoning": response_text
    })

eval_report = evaluate_submission(responses, problems)
print(f"Baseline Overall Accuracy: {eval_report['overall_accuracy'] * 100:.2f}%")

output_dir = config.OUTPUT_DIR
output_dir.mkdir(parents=True, exist_ok=True)
with open(output_dir / "baseline_results.json", "w") as f:
    json.dump(eval_report, f, indent=2)
print("Saved baseline_results.json")

In [ ]:
# Cell 6: Failure Mode Analysis
from collections import Counter

failure_modes = []
for p, r in zip(problems, responses):
    is_correct = check_answer(r["answer"], p["answer"])
    if not is_correct:
        tag = classify_failure(r)
        failure_modes.append(tag)

if not failure_modes:
    print("WARNING: No failures detected — all answers correct")
else:
    counts = Counter(failure_modes)
    print("Failure Mode Distribution:")
    for k, v in counts.items():
        print(f"  {k}: {v} errors")

failure_examples = {}
for p, r in zip(problems, responses):
    is_correct = check_answer(r["answer"], p["answer"])
    if not is_correct:
        mode = classify_failure(r)
        if mode not in failure_examples:
            failure_examples[mode] = []
        failure_examples[mode].append(p)

failure_data = {
    "failure_counts": dict(Counter(failure_modes)),
    "failure_examples": failure_examples,
}
with open(output_dir / "failure_modes.json", "w") as f:
    json.dump(failure_data, f, indent=2)
print(f"Saved failure_modes.json with {len(failure_examples)} failure modes")

In [ ]:
# Cell 7: Synthetic Data Generation
import os
from src.data.synthetic_generator import SyntheticGenerator

api_key = os.environ.get("TOGETHER_API_KEY")
if not api_key:
    raise ValueError(
        "TOGETHER_API_KEY environment variable is required. "
        "Set it via Kaggle Secrets or environment."
    )

generator = SyntheticGenerator(
    api_key=api_key,
    output_dir=str(config.OUTPUT_DIR)
)

if not failure_examples:
    print("No failure examples to generate from. Skipping generation.")
    raw_synthetic = []
else:
    raw_synthetic = generator.generate_per_failure_mode(
        failure_examples=failure_examples,
        problems_per_mode=config.SYNTHETIC_TARGET // max(len(failure_examples), 1)
    )

generator.save_dataset(raw_synthetic, filename="raw_synthetic_dataset.jsonl")

In [ ]:
# Cell 8: Quality Filtering & Deduplication
from src.data.judge_filter import JudgeFilter
from src.data.deduplicator import Deduplicator
from src.data.dataset_mixer import DatasetMixer

print("Step 1: Running Judge Filter (composite score validation)...")
judge = JudgeFilter(threshold=0.80)
filtered = judge.filter_dataset(raw_synthetic)
print(f"Filtered dataset from {len(raw_synthetic)} to {len(filtered)} items.")

print("\nStep 2: Running Deduplicator (MinHash + LSH)...")
dedup = Deduplicator(similarity_threshold=0.85)
deduplicated = dedup.deduplicate(filtered, key="question")
print(f"Deduplicated dataset to {len(deduplicated)} items.")

print("\nStep 3: Loading OpenMathReasoning and OpenCodeReasoning...")
import json as _json
from pathlib import Path as _Path

_pipeline_cfg = _json.loads(_Path("configs/pipeline.json").read_text())
from src.data.dataset_sources import load_openmath_reasoning, load_open_code_reasoning

openmath_data = load_openmath_reasoning(
    kaggle_path=_pipeline_cfg["openmath_kaggle_path"],
    hf_dataset=_pipeline_cfg["openmath_hf_dataset"],
    hf_split=_pipeline_cfg["openmath_hf_split"],
    max_samples=_pipeline_cfg["openmath_max_samples"],
)
opencode_data = load_open_code_reasoning(
    kaggle_path=_pipeline_cfg["opencode_kaggle_path"],
    hf_dataset=_pipeline_cfg["opencode_hf_dataset"],
    hf_split=_pipeline_cfg["opencode_hf_split"],
    max_samples=_pipeline_cfg["opencode_max_samples"],
)
print(f"Loaded {len(openmath_data)} OpenMath + {len(opencode_data)} OpenCode examples")

print("\nStep 4: Mixing Datasets (50/25/25 stratified)...")
mixer = DatasetMixer(seed=SEED)
final_dataset = mixer.mix(
    synthetic=deduplicated,
    math_reasoning=openmath_data,
    code_reasoning=opencode_data,
    max_total=50000,
)
print(f"Final mixed dataset: {len(final_dataset)} examples")

In [ ]:
# Cell 9: Leakage Check
test_questions = [p["question"] for p in problems]

def get_5grams(text: str) -> set:
    words = re.findall(r'\w+', text.lower())
    return set(tuple(words[i:i+5]) for i in range(len(words)-4))

test_5grams = set()
for q in test_questions:
    test_5grams.update(get_5grams(q))

leakage_found = False
for i, ex in enumerate(final_dataset):
    ex_5grams = get_5grams(ex["question"])
    intersection = test_5grams.intersection(ex_5grams)
    if intersection:
        print(f"WARNING: Potential leakage found in item {i}: {intersection}")
        leakage_found = True

if not leakage_found:
    print("\u2713 Leakage check passed: 0 overlapping 5-grams with test set.")

In [ ]:
# Cell 10: Save final dataset
final_output = config.FINAL_PATH
with open(final_output, "w") as f:
    for item in final_dataset:
        f.write(json.dumps(item) + "\n")
print(f"Saved {len(final_dataset)} final mixed examples to {final_output}")

In [ ]:
# Cell 11: Cleanup
import gc
if 'model' in globals() and model is not None:
    del model
torch.cuda.empty_cache()
gc.collect()
print("GPU memory cleared.")
print('P1 Complete — Phase gate: python scripts/verify_unit_completion.py P1')